# Give Your CAL Agent Access to Any MCP Server

[Model Context Protocol (MCP)](https://modelcontextprotocol.io/) lets AI agents
connect to external tools and data sources through a standard interface. Instead
of writing custom tool functions, you point your agent at an MCP server and it
automatically discovers and uses whatever tools that server exposes.

This notebook walks through connecting a CAL agent to the
[Context7](https://context7.com) MCP server, which provides up-to-date library
documentation lookups. By the end you'll know how to wire **any** MCP server
into a CAL agent.

### Architecture

```
                          CAL Agent
                     +-----------------+
  User Query ------> |   GeminiLLM     |
                     |   + Memory      |
                     +--------+--------+
                              |
              +---------------+---------------+
              |               |               |
       +------+------+ +-----+------+ +------+------+
       | resolve-    | | get-library| |    stop     |
       | library-id  | | -docs      | | (CAL Tool)  |
       | (MCP Tool)  | | (MCP Tool) | +-------------+
       +------+------+ +-----+------+
              |               |
              +-------+-------+
                      |
               +------+------+
               |  Context7   |
               |  MCP Server |
               | (subprocess)|
               +-------------+
```

### Prerequisites

1. **Node.js / npx** — Context7 runs as an MCP server via `npx`
2. **Gemini API key** — set `GEMINI_API_KEY` in a `.env` file or as an environment variable
3. **Install CAL with MCP extras:**

```bash
pip install "creevo-agent-library[mcp] @ git+https://github.com/Creevo-App/creevo-agent-library.git"
```

In [ ]:
# Uncomment to install dependencies
# !pip install "creevo-agent-library[mcp] @ git+https://github.com/Creevo-App/creevo-agent-library.git"

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads GEMINI_API_KEY from .env

# Or set it directly:
# os.environ["GEMINI_API_KEY"] = "your-key-here"

assert os.getenv("GEMINI_API_KEY"), "Set GEMINI_API_KEY in .env or as an environment variable"

---
## Step 1 — Connect to the MCP Server

`connect_mcp_server()` starts an MCP server as a child process and returns the
tools it exposes as `MCPTool` objects. These work just like any other CAL tool —
you can pass them straight into an `Agent`.

The only things you need are the **command** and **args** to launch the server
(the same values you'd put in an MCP client config).

In [ ]:
from CAL.mcp import connect_mcp_server, disconnect_mcp_tools

mcp_tools = await connect_mcp_server(
    command="npx",
    args=["-y", "@upstash/context7-mcp"],
)

print(f"Connected — discovered {len(mcp_tools)} tool(s):\n")
for t in mcp_tools:
    print(f"  - {t.name}: {t.description[:90]}")

Each `MCPTool` exposes the same JSON schema interface that `@tool` and
`@subagent` use. Schemas from MCP servers are automatically sanitized so
they're compatible with Gemini's function-calling format.

In [ ]:
import json

for t in mcp_tools:
    schema = t.get_schema()
    print(f"── {schema['name']} ──")
    print(json.dumps(schema["input_schema"], indent=2))
    print()

---
## Step 2 — Create the Agent

Build a standard CAL `Agent` and pass the MCP tools alongside `StopTool`.
No special wiring is needed — `MCPTool` implements the same `Tool` interface
as `@tool` and `@subagent`, so the agent treats them identically.

Key parameters:
- **`tools`** — mix native CAL tools and MCP tools in a single list
- **`max_calls`** — caps the total number of tool-call iterations
- **`memory_engine`** — `DefaultMemoryEngine` manages conversation history with
  multi-layer memory (recent turns, working memory, semantic recall, archives)

In [ ]:
from CAL import Agent, GeminiLLM, StopTool, DefaultMemoryEngine

api_key = os.getenv("GEMINI_API_KEY")

llm = GeminiLLM(model="gemini-3-flash-preview", api_key=api_key, max_tokens=4096)
summarizer_llm = GeminiLLM(model="gemini-3-flash-preview", api_key=api_key, max_tokens=2048)

agent = Agent(
    llm=llm,
    system_prompt=(
        "You are a helpful coding assistant. "
        "Use the Context7 MCP tools to look up library documentation before answering. "
        "Always cite the library version you referenced. "
        "Call stop when you have a complete answer."
    ),
    max_calls=15,
    max_tokens=4096,
    memory_engine=DefaultMemoryEngine(summarizer_llm=summarizer_llm),
    agent_name="context7-agent",
    tools=[StopTool(), *mcp_tools],
)

print(f"Agent ready — {len(agent.tools)} tools registered:")
for t in agent.tools:
    print(f"  - {t.name}")

---
## Step 3 — Run a Query

When the agent receives a question, it will autonomously:
1. Call `resolve-library-id` to find Context7's internal ID for the library
2. Call `get-library-docs` to fetch relevant documentation pages
3. Synthesize an answer from the docs and call `stop`

Before running, let's set up a small helper to display what happened.

In [ ]:
from CAL.content_blocks import TextBlock, ToolUseBlock, ToolResultBlock
from CAL.message import MessageRole


def show_run(agent):
    """Print the tool-call trace followed by the agent's final text answer."""
    # Tool trace
    step = 0
    for msg in agent.memory.get_history():
        if not isinstance(msg.content, list):
            continue
        for block in msg.content:
            if isinstance(block, ToolUseBlock):
                step += 1
                args = json.dumps(block.input, ensure_ascii=False)
                if len(args) > 80:
                    args = args[:77] + "..."
                print(f"  [{step}] {block.name}({args})")

    # Final answer — last assistant text blocks before 'stop'
    text_parts = []
    for msg in agent.memory.get_history():
        if msg.role == MessageRole.ASSISTANT and isinstance(msg.content, list):
            for block in msg.content:
                if isinstance(block, TextBlock) and block.text.strip():
                    text_parts.append(block.text)
    print("\n" + "=" * 60)
    if text_parts:
        # The last 1-2 text parts contain the synthesized answer
        print("\n".join(text_parts[-2:]))
    else:
        print("(No text response found)")

In [ ]:
result = await agent.run_async(
    "What does React useEffect do? Give a brief explanation with a small code example."
)

show_run(agent)

---
## Step 4 — Follow-up Query (Conversation Memory)

The agent keeps conversation history via `DefaultMemoryEngine`, so follow-up
questions work without re-explaining context. The agent may skip the
`resolve-library-id` step this time since it already knows the Context7 ID.

In [ ]:
result = await agent.run_async(
    "Now show me how to use useEffect with a cleanup function."
)

show_run(agent)

---
## Step 5 — Inspect the Full Conversation Trace

Every message that passed through the agent loop is stored in memory.
This is useful for debugging or understanding the agent's reasoning.

In [ ]:
ROLE_LABEL = {MessageRole.USER: "USER ", MessageRole.ASSISTANT: "AGENT"}

for msg in agent.memory.get_history():
    label = ROLE_LABEL.get(msg.role, "???  ")
    if isinstance(msg.content, str):
        print(f"  {label} | {msg.content[:100]}")
    elif isinstance(msg.content, list):
        for block in msg.content:
            if isinstance(block, TextBlock):
                preview = block.text[:80].replace("\n", " ")
                suffix = "..." if len(block.text) > 80 else ""
                print(f"  {label} | text: {preview}{suffix}")
            elif isinstance(block, ToolUseBlock):
                print(f"  {label} | call: {block.name}({json.dumps(block.input)[:60]})")
            elif isinstance(block, ToolResultBlock):
                status = "error" if block.is_error else "ok"
                print(f"  {label} | result({block.name}): [{status}]")

---
## Step 6 — Cleanup

`disconnect_mcp_tools()` shuts down the MCP server subprocess. Always call
this when you're done — otherwise the child process keeps running in the
background.

In [ ]:
await disconnect_mcp_tools(mcp_tools)
print("MCP server disconnected.")

---
## Connect Your Own MCP Server

Swap in any MCP server by changing the `command` and `args`. For example:

```python
# Filesystem server
tools = await connect_mcp_server(
    command="npx",
    args=["-y", "@modelcontextprotocol/server-filesystem", "/path/to/dir"],
)

# A local Python MCP server
tools = await connect_mcp_server(
    command="python",
    args=["-m", "my_mcp_server"],
    env={"MY_API_KEY": "..."},          # pass env vars to the subprocess
)

# Pass env vars to servers that need API keys
tools = await connect_mcp_server(
    command="npx",
    args=["-y", "@some-org/some-mcp-server"],
    env={"API_KEY": os.getenv("SOME_API_KEY")},
)
```

You can connect to **multiple** MCP servers and combine their tools in one agent:

```python
context7_tools = await connect_mcp_server(command="npx", args=["-y", "@upstash/context7-mcp"])
fs_tools = await connect_mcp_server(command="npx", args=["-y", "@modelcontextprotocol/server-filesystem", "."])

agent = Agent(
    ...,
    tools=[StopTool(), *context7_tools, *fs_tools],
)

# Clean up all connections when done
await disconnect_mcp_tools(context7_tools)
await disconnect_mcp_tools(fs_tools)
```